# MIT805 Group Project - Group 19
## Part 1 - Exploratory Data Analysis

### Imports and Setup

In [14]:
# Colab setup

#from google.colab import drive
#drive.mount('/content/drive')

In [15]:
# Imports

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pyarrow.parquet as pq

from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, IntegerType, DoubleType, TimestampType)


In [16]:
# Paths

PROJECT_ROOT = Path.cwd()

RAW_DIR   = PROJECT_ROOT / "data" / "working"
CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned_monthly"
FIG_DIR   = PROJECT_ROOT / "figures"
OUT_DIR   = PROJECT_ROOT / "output"

FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

spark = (SparkSession.builder
         .appName("MIT805 Part1 EDA")
         .config("spark.driver.memory", "6g")
         .config("spark.sql.shuffle.partitions", "24")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

STATS = {}

print("project:", PROJECT_ROOT)
print("Spark  :", spark.version)

project: /Users/shreyabharat/Projects/MIT805-GroupProject
Spark  : 4.2.0


In [17]:
# Figure style

BLUE = "#0000ff"
GREEN = "#00963f"
GREY =  "#52514e"

plt.rcParams.update({
    "figure.dpi": 200,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.titleweight": "semibold",
    "axes.titlelocation": "center",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "axes.axisbelow": True
})

def thousands(x, _=None):
    if x >= 1e6: return f"{x/1e6:.0f}M"
    if x >= 1e3: return f"{x/1e3:.0f}k"
    return f"{x:.0f}"

def save(fig, name):
    fig.savefig(FIG_DIR / name)
    plt.close(fig)
    print("saved:", name)

### Data loading

In [18]:
raw_files = sorted(RAW_DIR.glob("yellow_tripdata_*.parquet"))
month_dirs = sorted(d for d in CLEAN_DIR.glob("yellow_tripdata_*_cleaned")
                    if any(d.glob("*.parquet")))

months = [d.name.replace("yellow_tripdata_","").replace("_cleaned","")
          for d in month_dirs]
paths = [str(d) for d in month_dirs]

print(f"downloaded: {len(raw_files)} months of data")
print(f"cleaned: {len(months)} months, {months[0]} to {months[-1]}")

missing_data = len(raw_files) - len(months)
if missing_data:
    print(f"{missing_data} downloaded data files that did not have clean rows")

STATS["months_total_raw"] = len(raw_files)
STATS["months_with_data"] = len(months)
STATS["partial_run"] = bool(missing_data)

SCHEMA = StructType([
    StructField("tpep_pickup_datetime",TimestampType()),
    StructField("trip_distance",DoubleType()),
    StructField("pulocationid",IntegerType()),
    StructField("fare_amount",DoubleType()),
    StructField("tip_amount",DoubleType()),
    StructField("total_amount",DoubleType()),
    StructField("passenger_count",IntegerType()),
    StructField("trip_duration_min",DoubleType()),
    StructField("pickup_year",IntegerType()),
    StructField("pickup_month",IntegerType()),
    StructField("pickup_hour",IntegerType()),
    StructField("pickup_dayofweek",IntegerType()),
])

df = spark.read.schema(SCHEMA).parquet(*paths)
n = df.count()

processing_gb = sum(f.stat().st_size for p in paths for f in Path(p).rglob("*.parquet")) / 1024**3

raw_rows = sum(pq.ParquetFile(f).metadata.num_rows for f in raw_files)

STATS.update(
    analysis_rows=int(n),
    processing_gb=round(processing_gb, 2),
    raw_rows_total=int(raw_rows),
    retention_pct_overall=round(n / raw_rows * 100, 2),
    working_set_gb=round(sum(f.stat().st_size for f in raw_files) / 1024**3, 2)
    )

print(f"retention: {STATS['retention_pct_overall']}% of {raw_rows:,} raw rows")
print(f"processing tier: {processing_gb:.2f} GB ")

downloaded: 139 months of data
cleaned: 139 months, 2014-06 to 2025-12
retention: 95.85% of 895,333,567 raw rows
processing tier: 18.77 GB 


### Data quality

In [19]:
# Filter out dates not in selected range

year_min = 2014
year_max = 2025

invalid = df.filter(~F.col("pickup_year").between(year_min, year_max))
STATS["invalid_dates"] = invalid.count()

df = df.filter(F.col("pickup_year").between(year_min, year_max))
n = df.count()
print(f"rows after date filter: {n:,}({STATS['analysis_rows'] - n:,} removed)")

rows after date filter: 858,197,016(2,322 removed)


In [20]:
# Data distribution

percentiles = [0.01, 0.25, 0.5, 0.75, 0.95, 0.99]

for col in ["trip_distance", "fare_amount", "total_amount", "trip_duration_min"]:
    vals = df.approxQuantile(col, percentiles, 0.001)
    STATS[f"q_{col}"] = {f"p{int(q*100)}": round(v, 2) for q, 
                         v in zip(percentiles, vals)}

summary = df.agg(
    F.min("tpep_pickup_datetime").alias("first_pickup"),
    F.max("tpep_pickup_datetime").alias("last_pickup"),
    F.avg("trip_distance").alias("mean_distance"),
    F.avg("fare_amount").alias("mean_fare"),
    F.avg("tip_amount").alias("mean_tip"),
    F.count_distinct("pulocationid").alias("distinct_pu_zones"),
).collect()[0].asDict()

STATS.update(
    first_pickup = str(summary["first_pickup"].date()),
    last_pickup = str(summary["last_pickup"].date()),
    mean_distance = round(summary["mean_distance"], 2),
    mean_fare = round(summary["mean_fare"], 2),
    distinct_pu_zones = int(summary["distinct_pu_zones"]),
    tip_rate_pct = round(summary["mean_tip"] / summary["mean_fare"] * 100, 2),
)


### Plots

In [21]:
# Fig 1: Monthly trip volume

monthly = (df.groupBy("pickup_year", "pickup_month")
             .agg(F.count(F.lit(1)).alias("trips"))
             .orderBy("pickup_year", "pickup_month")
             .toPandas()
             )

monthly["date"] = pd.to_datetime(dict(year=monthly.pickup_year, month=monthly.pickup_month, day=1))
monthly.to_csv(OUT_DIR / "monthly_volume.csv", index=False)

peak   = monthly.loc[monthly.trips.idxmax()]
lowest = monthly.loc[monthly.trips.idxmin()]
latest = monthly.iloc[-1]

STATS["monthly_peak"] = {"month": f"{peak.date:%Y-%m}", "trips": int(peak.trips)}
STATS["monthly_lowest"] = {"month": f"{lowest.date:%Y-%m}", "trips": int(lowest.trips)}
STATS["monthly_latest"] = {"month": f"{latest.date:%Y-%m}", "trips": int(latest.trips)}
STATS["covid_drop_pct"] = round((1 - lowest.trips / peak.trips) * 100, 1)
STATS["pct_of_peak"] = round(latest.trips / peak.trips * 100, 1)

print(f"peak:{peak.date:%Y-%m} = {int(peak.trips):,}")
print(f"lowest:{lowest.date:%Y-%m} = {int(lowest.trips):,} "f"({STATS['covid_drop_pct']}% below peak)")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(monthly.date, monthly.trips, color=BLUE, linewidth=2)
ax.set_title(f"Monthly trip volume")
ax.set_ylabel("Trips per month")
ax.set_ylim(0, monthly.trips.max() * 1.18)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(thousands))
save(fig, "fig1_monthly_volume.png")

peak:2014-10 = 14,224,018
lowest:2020-04 = 205,234 (98.6% below peak)
saved: fig1_monthly_volume.png


In [22]:
# Fig 2: Hourly demand (weekdays vs weekends)

hourly = (df.withColumn("weekend", F.col("pickup_dayofweek").isin(1, 7))
            .groupBy("pickup_hour", "weekend")
            .agg(F.count(F.lit(1)).alias("trips"),
                 F.avg(F.col("trip_distance")
                       / (F.col("trip_duration_min") / 60)).alias("mph"))
            .orderBy("pickup_hour")
            .toPandas()
            )

weekday = hourly[~hourly.weekend].copy()
weekend = hourly[hourly.weekend].copy()
weekday["share"] = weekday.trips / weekday.trips.sum() * 100
weekend["share"] = weekend.trips / weekend.trips.sum() * 100

pd.concat([weekday, weekend]).to_csv(OUT_DIR / "hourly_profile.csv", index=False)

fastest = weekday.loc[weekday.mph.idxmax()]
slowest = weekday.loc[weekday.mph.idxmin()]

STATS["peak_hour_weekday"] = int(weekday.loc[weekday.share.idxmax(), "pickup_hour"])
STATS["peak_hour_weekend"] = int(weekend.loc[weekend.share.idxmax(), "pickup_hour"])
STATS["fastest_hour_mph"] = {"hour": int(fastest.pickup_hour),"mph": round(fastest.mph, 2)}
STATS["slowest_hour_mph"] = {"hour": int(slowest.pickup_hour),"mph": round(slowest.mph, 2)}

print(f"peak hour: {STATS['peak_hour_weekday']}:00 weekday, "f"{STATS['peak_hour_weekend']}:00 weekend")
print(f"speed: {fastest.mph:.1f} mph at {int(fastest.pickup_hour):02d}:00, "f"{slowest.mph:.1f} mph at {int(slowest.pickup_hour):02d}:00")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(weekday.pickup_hour, weekday.share, color=BLUE, linewidth=2, label="Weekday")
ax.plot(weekend.pickup_hour, weekend.share, color=GREEN, linewidth=2, label="Weekend")
ax.set_title("Average hourly demand (weekdays vs weekends)")
ax.set_xlabel("Hour of pickup")
ax.set_ylabel("% of trips")
ax.set_xticks(range(0, 24, 3))
ax.set_xlim(0, 23)
ax.set_ylim(0)
ax.legend()
save(fig, "fig2_hourly_demand.png")

peak hour: 18:00 weekday, 18:00 weekend
speed: 25.5 mph at 04:00, 11.2 mph at 11:00
saved: fig2_hourly_demand.png


In [23]:
# Fig 3: Trip distances

bin_width = 0.25
p99 = STATS["q_trip_distance"]["p99"]

dist = (df.filter(F.col("trip_distance") <= p99)
          .withColumn("bin", F.floor(F.col("trip_distance") / bin_width) * bin_width)
          .groupBy("bin")
          .agg(F.count(F.lit(1)).alias("trips"))
          .orderBy("bin")
          .toPandas())

dist["share"] = dist.trips / dist.trips.sum() * 100
dist.to_csv(OUT_DIR / "distance_distribution.csv", index=False)

STATS["pct_trips_under_3miles"] = round(dist[dist.bin < 3].share.sum(), 1)

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(dist.bin, dist.share, width=bin_width * 0.86, color=BLUE)
ax.set_title("Trip distance distribution")
ax.set_xlabel("Trip distance (miles)")
ax.set_ylabel("% of trips")
ax.set_xlim(0, p99)
save(fig, "fig3_distance_distribution.png")

saved: fig3_distance_distribution.png


In [24]:
# Fig 4: Pickup zone concentration

ZONE_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zones = pd.read_csv(ZONE_URL)
zones.columns = [c.lower() for c in zones.columns]

zone_counts = (df.groupBy("pulocationid")
                 .agg(F.count(F.lit(1)).alias("trips"))
                 .orderBy(F.desc("trips"))
                 .toPandas())

zone_counts = zone_counts.merge(zones[["locationid", "zone", "borough"]],
                                left_on="pulocationid",
                                right_on="locationid",
                                how="left")
zone_counts["share"] = zone_counts.trips / zone_counts.trips.sum() * 100
zone_counts.to_csv(OUT_DIR / "zone_counts.csv", index=False)

top10 = zone_counts.head(10)
borough_share = zone_counts.groupby("borough").share.sum().sort_values(ascending=False)

STATS["top10_zone_share"] = round(top10.share.sum(), 1)
STATS["top_zone"] = {"zone": top10.iloc[0].zone,"borough": top10.iloc[0].borough,"share": round(top10.iloc[0].share, 2)}
STATS["borough_share"] = borough_share.round(1).to_dict()

print(f"top 10 zones = {STATS['top10_zone_share']}% of pickups")
print(borough_share.round(1).to_string())

top15 = zone_counts.head(15).iloc[::-1]

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top15.zone, top15.share, height=0.72, color=BLUE)

for i, share in enumerate(top15.share):
    ax.text(share + 0.1, i, f"{share:.1f}%", va="center", fontsize=7.5, color=GREY)

ax.set_title("Top pickup zones")
ax.set_xlabel("% of all pickups")
ax.set_xlim(0, top15.share.max() * 1.15)
save(fig, "fig4_top_zones.png")

top 10 zones = 34.1% of pickups
borough
Manhattan        90.6
Queens            6.5
Brooklyn          1.4
Unknown           1.3
Bronx             0.1
Staten Island     0.0
EWR               0.0
saved: fig4_top_zones.png


### Summary

In [25]:
# Velocity
daily = (df.groupBy(F.to_date("tpep_pickup_datetime").alias("day"))
           .agg(F.count(F.lit(1)).alias("trips"))
           .toPandas()
           .dropna())

busiest = daily.loc[daily.trips.idxmax()]

STATS["mean_trips_per_day"] = int(daily.trips.mean())
STATS["busiest_day"] = {"date": str(busiest.day), "trips": int(busiest.trips)}
STATS["peak_trips_per_min"] = round(busiest.trips / 1440, 1)

print(f"mean {STATS['mean_trips_per_day']:,} trips/day")
print(f"busiest {busiest.day}: {int(busiest.trips):,} trips "f"({STATS['peak_trips_per_min']}/min)")

# Variety
schemas = {}
for f in raw_files:
    month = f.stem.replace("yellow_tripdata_", "")
    schemas[month] = {c.lower() for c in pq.ParquetFile(f).schema.names}

oldest_month = min(schemas)
newest_month = max(schemas)
oldest = schemas[oldest_month]
newest = schemas[newest_month]

STATS["schema_distinct_layouts"] = len({frozenset(s) for s in schemas.values()})
STATS["schema_added"] = sorted(newest - oldest)
STATS["schema_removed"] = sorted(oldest - newest)

schema_table = pd.DataFrame({"month": sorted(schemas),
                             "n_cols": [len(schemas[m]) for m in sorted(schemas)]})

schema_table.to_csv(OUT_DIR / "schema_by_month.csv", index=False)

print(f"{oldest_month}: {len(oldest)} columns, {newest_month}: {len(newest)} columns")
print(f"{STATS['schema_distinct_layouts']} distinct layouts across {len(schemas)} files")
print("added:", STATS["schema_added"])
print("removed:", STATS["schema_removed"])

# Value
revenue_table = (df.filter(F.col("trip_duration_min") >= 1)
               .groupBy("pickup_hour")
               .agg(F.count(F.lit(1)).alias("trips"),
                    (F.sum("total_amount")
                     / F.sum(F.col("trip_duration_min") / 60)).alias("rev_per_veh_hour"))
               .orderBy("pickup_hour")
               .toPandas())

revenue_table.to_csv(OUT_DIR / "hourly_yield.csv", index=False)

best = revenue_table.loc[revenue_table.rev_per_veh_hour.idxmax()]
worst = revenue_table.loc[revenue_table.rev_per_veh_hour.idxmin()]

STATS["yield_best_hour"] = {"hour": int(best.pickup_hour),
                            "usd_per_veh_hour": round(best.rev_per_veh_hour, 2)}
STATS["yield_worst_hour"] = {"hour": int(worst.pickup_hour),
                             "usd_per_veh_hour": round(worst.rev_per_veh_hour, 2)}
STATS["yield_spread_pct"] = round(
    (best.rev_per_veh_hour / worst.rev_per_veh_hour - 1) * 100, 1)

print(f"best {int(best.pickup_hour):02d}:00 = ${best.rev_per_veh_hour:.2f}/veh-hr")
print(f"worst {int(worst.pickup_hour):02d}:00 = ${worst.rev_per_veh_hour:.2f}/veh-hr")
print(f"spread: {STATS['yield_spread_pct']}%")

for key, value in STATS.items():
    print(f"{key}: {value}")

mean 202,739 trips/day
busiest 2014-11-01: 574,485 trips (398.9/min)
2014-06: 19 columns, 2025-12: 20 columns
2 distinct layouts across 139 files
added: ['cbd_congestion_fee']
removed: []


best 05:00 = $88.35/veh-hr
worst 15:00 = $60.29/veh-hr
spread: 46.6%
months_total_raw: 139
months_with_data: 139
partial_run: False
analysis_rows: 858199338
processing_gb: 18.77
raw_rows_total: 895333567
retention_pct_overall: 95.85
working_set_gb: 12.15
invalid_dates: 2322
q_trip_distance: {'p1': 0.3, 'p25': 1.0, 'p50': 1.7, 'p75': 3.2, 'p95': 11.15, 'p99': 19.05}
q_fare_amount: {'p1': 3.5, 'p25': 7.0, 'p50': 10.0, 'p75': 15.5, 'p95': 39.5, 'p99': 62.5}
q_total_amount: {'p1': 4.8, 'p25': 9.5, 'p50': 13.56, 'p75': 20.3, 'p95': 51.34, 'p99': 80.87}
q_trip_duration_min: {'p1': 1.7, 'p25': 6.87, 'p50': 11.38, 'p75': 18.53, 'p95': 37.23, 'p99': 61.43}
first_pickup: 2014-06-01
last_pickup: 2025-12-31
mean_distance: 3.06
mean_fare: 13.83
distinct_pu_zones: 265
tip_rate_pct: 15.21
monthly_peak: {'month': '2014-10', 'trips': 14224018}
monthly_lowest: {'month': '2020-04', 'trips': 205234}
monthly_latest: {'month': '2025-12', 'trips': 2946248}
covid_drop_pct: 98.6
pct_of_peak: 20.7
peak_hour_wee